In [14]:
import json
import base64
from langchain.tools import tool, ToolRuntime
from typing import Annotated
from api.v1 import state
from utils.transport_utils import enrich_info_with_thresholds
@tool
def get_roads() -> str:
    """Lấy danh sách tên các tuyến đường đang được analyzer quản lý.

    Returns:
        str: Chuỗi JSON chứa một trong các dạng:
            - {"error": "..."} khi analyzer chưa sẵn sàng.
            - {"roads": [], "message": "..."} khi chưa có tuyến đường.
            - {"roads": ["..."]} khi có dữ liệu tuyến đường.
    """
    if state.analyzer is None:
        return json.dumps({"error": "Analyzer chưa được khởi tạo"}, ensure_ascii=False)
    
    road_names = state.analyzer.names
    if not road_names:
        return json.dumps({"roads": [], "message": "Không có tuyến đường nào."}, ensure_ascii=False)
    
    return json.dumps({"roads": road_names}, ensure_ascii=False)
    
@tool
def get_frame_road(
    road_name: Annotated[str, "Tên tuyến đường"],
    runtime: ToolRuntime,
) -> str:
    """Lấy frame hiện tại của một tuyến đường và lưu ảnh vào private state theo thread.

    Args:
        road_name (Annotated[str, "Tên tuyến đường"]): Tên tuyến đường cần lấy ảnh.
        runtime (ToolRuntime): Runtime của tool để đọc `thread_id` từ config.

    Returns:
        str: Thông báo kết quả xử lý (thành công hoặc lỗi).
    """
    try:
        if state.analyzer is None:
            return "Analyzer chưa được khởi tạo, không thể lấy ảnh."

        frame_bytes = state.analyzer.get_frame_road(road_name)
        if not frame_bytes:
            return f"Không có frame hiện tại cho tuyến đường '{road_name}'."

        configurable = runtime.config.get("configurable", {})
        thread_id = str(configurable.get("thread_id", "anonymous"))
        frame_b64 = base64.b64encode(frame_bytes).decode("ascii")
        state.append_private_image(thread_id, frame_b64)
        return f"Đã lấy ảnh hiện tại cho tuyến đường '{road_name}'."
    except Exception as e:
        return f"Lỗi không xác định: {str(e)}"

@tool
def get_info_road(road_name: Annotated[str, "Tên tuyến đường"]) -> str:
    """Lấy thông tin giao thông của một tuyến đường dưới dạng JSON.

    Dữ liệu thô từ analyzer sẽ được enrich thêm theo ngưỡng cấu hình
    trước khi trả về cho agent.

    Args:
        road_name (Annotated[str, "Tên tuyến đường"]): Tên tuyến đường cần truy vấn.

    Returns:
        str: Chuỗi JSON chứa dữ liệu tuyến đường hoặc thông tin lỗi.
    """
    if state.analyzer is None:
        return json.dumps({"error": "Analyzer chưa được khởi tạo"}, ensure_ascii=False)
    
    data = state.analyzer.get_info_road(road_name)
    if not data:
        return json.dumps({"error": f"Không có dữ liệu cho tuyến đường '{road_name}'"}, ensure_ascii=False)
    data = enrich_info_with_thresholds(data, road_name)
    return json.dumps(data, ensure_ascii=False)
    

In [31]:

from langchain_google_genai import ChatGoogleGenerativeAI



LLM = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview",
                                temperature=0.6, 
                                max_output_tokens=1024
                                )

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [34]:
import dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from core.config import setting_chatbot
from langgraph.checkpoint.memory import InMemorySaver
from utils.chatbot_utils import TrimMessagesMiddleware
from api.v1 import state
from schemas.AgentTextResponse import AgentTextResponse


prompt = """Bạn là một trợ lý AI chuyên tư vấn giao thông bằng TIẾNG VIỆT.

MỤC TIÊU CHÍNH:
- Hiểu rõ ý định người dùng, trả lời ngắn gọn, chính xác và có cấu trúc.
- Khi người dùng yêu cầu thông tin về một hoặc nhiều tuyến đường, BẮT BUỘC phải cung cấp: số lượng và vận tốc trung bình của ô tô (ô tô) và xe máy (xe máy) cho từng tuyến và các thông tin về tình trạng giao thông của tuyến đường đó.

ĐỊNH DẠNG TRẢ LỜI (LUÔN BẰNG TIẾNG VIỆT):
1) Tóm tắt ngắn (1 câu)
2) Với mỗi tuyến đường được hỏi: tiêu đề tuyến ->
    - Số lượng ô tô: X
    - Vận tốc ô tô (trung bình): Y km/h
    - Số lượng xe máy: A
    - Vận tốc xe máy (trung bình): B km/h
    - Nhận xét tổng quát: (Ví dụ: Thông thoáng / Đông đúc / Tắc nghẽn)
3) Hành động khuyến nghị (2-3 gợi ý cụ thể, ví dụ chọn lộ trình, thời gian đi, cảnh báo)
4) Nếu người dùng yêu cầu ảnh: gọi `get_frame_road(road_name)` để hệ thống đính kèm ảnh qua API.
    KHÔNG in URL/base64 hay chuỗi dữ liệu ảnh vào phần `message`.

HƯỚNG DẪN HÀNH VI:
- Nếu người dùng không nói rõ tuyến đường, HỎI lại: "Bạn muốn thông tin tuyến đường nào?"
- Nếu có nhiều tuyến, trả lời theo mục rõ ràng cho từng tuyến.
- Tránh phán đoán không có dữ liệu; nếu thiếu dữ liệu, nói rõ: "Không có dữ liệu thời gian thực cho tuyến X" và gợi ý cách lấy (ví dụ: yêu cầu quyền, thử lại sau).
- Giữ giọng chuyên nghiệp, thân thiện và nhấn mạnh dữ liệu khi đưa khuyến nghị.

LƯU Ý KỸ THUẬT:
- Trả kết quả có thể parse được bởi chương trình (đặc biệt phần số liệu phải dễ trích xuất).
- Luôn trả bằng tiếng Việt.
"""
dotenv.load_dotenv()

class ChatBotAgent:
    def __init__(self):
        self.prompt = prompt
        self.checkpointer = InMemorySaver()
        self.agent = create_agent(
            model= LLM,
            tools=[get_frame_road, get_info_road, get_roads],
            system_prompt=prompt,
            response_format=ToolStrategy(AgentTextResponse),
            middleware=[TrimMessagesMiddleware(max_tokens=2000)],
            checkpointer=self.checkpointer,
        )

    
    async def get_response(self, user_input: str, id: int) -> dict:
        """Lấy phản hồi từ Agent dựa trên đầu vào của người dùng.

        Args:
            user_input (str): Nội dung tin nhắn của người dùng.

        Returns:
            dict: Phản hồi từ Agent, bao gồm hình ảnh và văn bản.
        """
        
        
        thread_id = f"{id}"
        state.clear_private_images(thread_id)
        config = {"configurable": {"thread_id": thread_id}}
        response = await self.agent.ainvoke(
            {"messages": [{"role": "user", "content": user_input}]},
            config = config
        )
        print("Raw agent response:", response)
        structured = response.get("structured_response")
        message = ""
        if structured is not None:
            message = getattr(structured, "message", "") or ""

        if not message:
            messages = response.get("messages", [])
            if messages:
                last_msg = messages[-1]
                message = getattr(last_msg, "content", "") or ""

        images = state.pop_private_images(thread_id)
        return {
            "message": message,
            "image": images,
        }

In [35]:

# ************ TESTING ************
if __name__ == "__main__":
    chat = ChatBotAgent()
    res = await chat.get_response("cho tôi xin thông tin về Văn Phú và Văn Quán, cả ảnh nữa nhé", id= 1)
    print(res)

Raw agent response: {'messages': [HumanMessage(content='cho tôi xin thông tin về Văn Phú và Văn Quán, cả ảnh nữa nhé', additional_kwargs={}, response_metadata={}, id='fea59733-a582-462e-97f3-22c27c264905'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_frame_road', 'arguments': '{"road_name": "V\\u0103n Qu\\u00e1n"}'}, '__gemini_function_call_thought_signatures__': {'b334f9fd-dfcd-455f-b5db-39ce3adc2ee5': 'EjQKMgG+Pvb7N7Y39w+L0ipa9yRsi03IJI2B2lgQmbuIy8lSTCncQu+edZ59QBKxwQD/USof'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d61ea-525b-7b92-976c-aac1a1dac56a-0', tool_calls=[{'name': 'get_info_road', 'args': {'road_name': 'Văn Phú'}, 'id': 'b334f9fd-dfcd-455f-b5db-39ce3adc2ee5', 'type': 'tool_call'}, {'name': 'get_info_road', 'args': {'road_name': 'Văn Quán'}, 'id': '3d826ff8-b8a4-4941-a9fb-52d5dddd11ae', 'type': 'tool_call'}, {'name': 'get_fra